# API参考: `HydraulicModel`

**类路径:** `preissmann_model.model.HydraulicModel`

## 目的
`HydraulicModel` 是**一维水动力模块**的核心。它代表一个独立的河段（Reach），并负责在该河段上求解一维圣维南方程（1D St. Venant equations），以模拟流量（`Q`）和水位（`Z`）随时间和空间的动态演变。

## 继承关系
`HydraulicModel` 继承自 `common.base_model.BaseModelComponent`，因此它可以作为一个标准组件被添加到`SimulationController`中，与其他水文或水力组件进行耦合。

## `__init__(self, name, reach, dt, downstream_level, structures=None, ...)`

构造函数用于创建一个`HydraulicModel`的实例。

**核心参数:**
- `name` (str): (继承自`BaseModelComponent`) 组件的唯一名称。
- `reach` (RiverReach): 一个定义了河段几何形状的`RiverReach`对象。
- `dt` (float): 模型内部进行数值计算时使用的时间步长 (秒)。
- `downstream_level` (float): **默认的**下游水位边界条件 (米)。这个值可以在模拟过程中被动态修改，例如当它连接到另一个河段或受潮汐影响时。
- `structures` (list, optional): 一个可选的列表，包含了位于该河段内的水工建筑物（如`Gate`或`Pump`）的实例。

### 示例

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection
from preissmann_model.structures import Gate

# 1. 创建子组件实例
reach_geom = RiverReach(
    cross_sections=[RectangularCrossSection(width=10) for _ in range(11)],
    lengths=np.full(10, 100.0), slope=0.001, manning_n=0.03
)
gate_structure = Gate(name="control_gate", node_index=5, opening_height=1.0, width=10)

# 2. 创建 HydraulicModel 实例
hydro_model = HydraulicModel(
    name="my_river_reach",
    reach=reach_geom,
    dt=10.0, # 10秒计算步长
    downstream_level=5.0, # 下游水位固定为5米
    structures=[gate_structure]
)

print(f"成功创建 HydraulicModel: '{hydro_model.name}'")

## `step(self, inflows: dict, dt: float)`

这是`HydraulicModel`作为`BaseModelComponent`所实现的接口方法，也是其核心计算方法。它负责组装并求解线性化的圣维南方程组，以计算出一个时间步之后新的`Q`和`Z`。

**逻辑流程:**
1.  根据当前时刻的`Q`和`Z`，计算出圣维南方程线性化后的系数矩阵`M`和右端项`R`。
2.  将上游边界条件（`Q_inflow`或`Z_inflow`）、下游边界条件和内部结构物（`structures`）的方程，更新到`M`和`R`的相应行中。
3.  调用`np.linalg.solve(M, R)`求解状态增量`dX`。
4.  将增量`dX`加到当前的`Q`和`Z`上，完成状态更新。
5.  更新自身的`outflow`属性（即最后一个节点的流量`self.Q[-1]`）。

**参数:**
- `inflows` (dict): 一个字典，包含了上游的边界条件。可以是：
    - `{'Q_inflow': float}`: 上游流量边界。
    - `{'Z_inflow': float}`: 上游水位边界。
    - 如果被其他组件连接，键可能是上游组件的名称。
- `dt` (float): 控制器传递的时间步长。**注意：** 此`dt`参数当前在`step`方法中**未被使用**，模型总是使用其在`__init__`中定义的内部`dt`。

### 示例

In [ ]:
# 定义上游入流
inflow_boundary = {'Q_inflow': 50.0} # 50 m³/s

print(f"运行前, 出口流量: {hydro_model.get_outflow():.3f} m³/s")

# 运行一步
hydro_model.step(inflows=inflow_boundary, dt=60) 

print(f"运行一步后, 出口流量: {hydro_model.get_outflow():.3f} m³/s")

## `run(self, num_steps, Q_inflow_hydrograph, Z_downstream_hydrograph)`

这是一个辅助方法，用于将`HydraulicModel`作为一个独立的、单机模型来运行，方便进行快速测试。它内部循环调用`step`方法。

**参数:**
- `num_steps` (int): 总模拟步数。
- `Q_inflow_hydrograph` (list or np.ndarray): 上游入流过程线。
- `Z_downstream_hydrograph` (list or np.ndarray): 下游水位过程线。

### 示例

In [ ]:
inflow_q = [50, 55, 60, 58, 52]
downstream_z = [5.0, 5.1, 5.2, 5.15, 5.05]

# 创建一个新实例用于演示
model_for_run = HydraulicModel("standalone_run", reach_geom, dt=10.0, downstream_level=5.0)
model_for_run.run(5, inflow_q, downstream_z)

print(f"通过 run() 运行5步后, 最终出口流量: {model_for_run.get_outflow():.3f} m³/s")